# Week 8: Naive Bayes Classifier
**Titanic Survival Prediction**

We'll use Naive Bayes to predict passenger survival based on features like class, sex, age, and fare.

## 1. Import Libraries

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Load & Explore Data
Using seaborn's built-in Titanic dataset.

In [30]:
# Load Titanic dataset
df = sns.load_dataset('titanic')

# Quick look at the data
print(f"Dataset shape: {df.shape}")
df.head()

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>

In [ ]:
# Check for missing values
df.isnull().sum()

## 3. Data Preprocessing
- Select relevant features
- Handle missing values
- Encode categorical variables

In [ ]:
# Select features for our model
features = ['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch']
target = 'survived'

# Create a copy with selected columns
data = df[features + [target]].copy()

# Handle missing values - fill age with median
data['age'].fillna(data['age'].median(), inplace=True)

# Encode sex: male=0, female=1
data['sex'] = data['sex'].map({'male': 0, 'female': 1})

# Drop any remaining NaN rows
data.dropna(inplace=True)

print(f"Clean data shape: {data.shape}")
data.head()

## 4. Split Data
80% training, 20% testing.

In [ ]:
# Separate features (X) and target (y)
X = data[features]
y = data[target]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 5. Train Naive Bayes Model
Using **GaussianNB** - assumes features follow a normal distribution.

In [ ]:
# Create and train the model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

print("Model trained!")

## 6. Make Predictions

In [ ]:
# Predict on test set
y_pred = nb_model.predict(X_test)

# Show first 10 predictions vs actual
comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10]
})
comparison

## 7. Evaluate Model

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

In [ ]:
# Confusion Matrix visualization
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Naive Bayes')
plt.tight_layout()
plt.show()

## 8. Predict New Passengers
Let's test with custom examples.

In [ ]:
# Create new passenger data
# Features: pclass, sex (0=male, 1=female), age, fare, sibsp, parch
new_passengers = pd.DataFrame({
    'pclass': [1, 3, 2],
    'sex': [1, 0, 1],      # female, male, female
    'age': [25, 30, 8],
    'fare': [100, 15, 50],
    'sibsp': [0, 1, 2],
    'parch': [0, 0, 1]
})

# Predict survival
predictions = nb_model.predict(new_passengers)
probabilities = nb_model.predict_proba(new_passengers)

# Display results
results = new_passengers.copy()
results['Survived'] = ['Yes' if p == 1 else 'No' for p in predictions]
results['Probability'] = [f"{prob[1]:.1%}" for prob in probabilities]
results

## Summary

**What we learned:**
- Naive Bayes uses Bayes' Theorem: P(A|B) = P(B|A) × P(A) / P(B)
- "Naive" assumption: features are independent
- Fast to train, works well for classification
- Good baseline model for comparison